In [ ]:

import os
import time

from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware


DEFAULT_CORS_ORIGINS = [
    "http://localhost:3000",
    "http://127.0.0.1:3000",
    "https://ai-study-companion-prototype.vercel.app",
]

# Vercel preview deployments get a new random URL per commit
# (https://<project>-<hash>-<team>.vercel.app). Set
# CORS_ALLOW_VERCEL_PREVIEWS=true to allow them; leave unset in
# strict production setups and list exact origins in CORS_ORIGINS.
VERCEL_PREVIEW_REGEX = r"https://[a-z0-9-]+\.vercel\.app"


def cors_origins() -> list[str]:
    """Allowed browser origins, including local frontend and configured origins."""

    raw = os.environ.get("CORS_ORIGINS", "")

    configured_origins = [
        origin.strip().rstrip("/")
        for origin in raw.split(",")
        if origin.strip()
    ]

    return list(dict.fromkeys(
        DEFAULT_CORS_ORIGINS + configured_origins
    ))


def cors_origin_regex() -> str | None:
    enabled = os.environ.get(
        "CORS_ALLOW_VERCEL_PREVIEWS", ""
    ).strip().lower() in {"1", "true", "yes"}

    return VERCEL_PREVIEW_REGEX if enabled else None


STARTED_AT = time.time()


def runtime_info() -> dict:
    """
    Non-secret runtime facts for /health.

    uptime_seconds is the quickest way to spot a crash/restart loop
    (it keeps resetting to a small number), and jwt_expire_minutes
    shows the token lifetime the *deployed* server is really using.
    """

    info = {
        "uptime_seconds": int(time.time() - STARTED_AT),
    }

    # Resident memory of this process. Render's free plan kills the
    # service above 512 MB, so this is the number to watch.
    try:
        with open("/proc/self/status", encoding="utf-8") as status_file:
            for line in status_file:
                if line.startswith("VmRSS:"):
                    info["memory_mb"] = round(int(line.split()[1]) / 1024)
                    break
    except OSError:
        pass

    try:
        from app.core.config import settings

        info["environment"] = settings.app_env
        info["jwt_expire_minutes"] = settings.jwt_expire_minutes
    except Exception:
        pass

    try:
        from app.services.ai_service import embedding_status

        info["embedding"] = embedding_status()
    except Exception:
        pass

    return info


def create_app() -> FastAPI:
    app = FastAPI(
        title="AI Study Companion",
        description=(
            "AI-powered learning and growth workspace "
            "with grounded tutoring, adaptive assessment "
            "and personalized learning."
        ),
        version="0.1.0",
    )

    app.add_middleware(
        CORSMiddleware,
        allow_origins=cors_origins(),
        allow_origin_regex=cors_origin_regex(),
        allow_credentials=True,
        allow_methods=["*"],
        allow_headers=["*"],
        expose_headers=["*"],
    )

    @app.get("/")
    async def root():
        return {
            "status": "ok",
            "service": "ai-study-companion",
        }

    @app.get("/health")
    async def health():
        return {
            "status": "ok",
            "service": "ai-study-companion",
            **runtime_info(),
        }

    return app


app = create_app()
